In [ ]:

import json
from html import escape
from ipywidgets import AppLayout, HTML, Output, VBox
from ipyleaflet import Map, DrawControl, Polygon, Popup, basemaps
from shapely.geometry import shape
from rasterio.mask import mask
import rasterio
import numpy as np
import geopandas as gpd
from collections import Counter


: 

In [ ]:

# Path to the merged raster file
LANDCOVER_PATH = "data/worldcover_2020_merged.tif"

# Optional: Define landcover class names
LANDCOVER_CLASSES = {
    10: "Trees",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare / sparse vegetation",
    70: "Snow and Ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss and lichen"
}


In [ ]:

output = Output()
DRAW_COLORS = ["#2f80ed", "#f2994a", "#27ae60", "#eb5757", "#9b51e0", "#00a7a7", "#f2c94c"]
drawn_areas = []
next_color_index = 0


def next_draw_color():
    return DRAW_COLORS[next_color_index % len(DRAW_COLORS)]


def set_draw_color(color):
    draw_control.polygon = {"shapeOptions": {"color": color, "fillColor": color, "fillOpacity": 0.45}}
    draw_control.rectangle = {"shapeOptions": {"color": color, "fillColor": color, "fillOpacity": 0.45}}

# Create map widget
m = Map(center=(0, 0), zoom=2, basemap=basemaps.Esri.WorldImagery, scroll_wheel_zoom=True, layout={'height': '50vh'})

# Draw control
draw_control = DrawControl(polyline={}, circlemarker={}, circle={}, marker={})
set_draw_color(next_draw_color())

# Add draw control to map
m.add_control(draw_control)


In [ ]:
# Replace the AppLayout cell with a full-screen map and output panel using VBox

from ipywidgets import VBox, Layout
from IPython.display import display

# Set the map height to 100vh for full screen
m.layout = Layout(height='100vh')

# Display the map and output panel vertically
display(VBox([m, output]))

In [ ]:
def polygon_locations(geom):
    return [(lat, lon) for lon, lat in geom.exterior.coords]


def build_popup(geom, title, lines):
    body = "".join(f"<div>{escape(line)}</div>" for line in lines)
    return Popup(
        location=(geom.centroid.y, geom.centroid.x),
        child=HTML(f"<strong>{escape(title)}</strong>{body}"),
        close_button=True,
        auto_close=False,
        close_on_escape_key=True,
        max_width=360,
    )


def show_popup(popup):
    if popup not in m.layers:
        m.add_layer(popup)


def add_analysis_layer(geom, color, popup):
    layer = Polygon(
        locations=polygon_locations(geom),
        color=color,
        fill_color=color,
        fill_opacity=0.35,
        weight=3,
    )

    def reopen_popup(**kwargs):
        show_popup(popup)

    layer.on_click(reopen_popup)
    drawn_areas.append({"layer": layer, "popup": popup, "color": color})
    m.add_layer(layer)
    show_popup(popup)


def handle_draw(target, action, geo_json):
    global next_color_index
    if action not in ("created", "edited"):
        return

    geom = shape(geo_json["geometry"])
    color = next_draw_color()
    output.clear_output()
    with output:
        try:
            gdf = gpd.GeoDataFrame(index=[0], crs="EPSG:4326", geometry=[geom])

            with rasterio.open(LANDCOVER_PATH) as src:
                raster_gdf = gdf.to_crs(src.crs) if src.crs else gdf
                raster_geom = json.loads(raster_gdf.to_json())["features"][0]["geometry"]
                clipped, transform = mask(src, [raster_geom], crop=True)
                data = clipped[0]
                nodata = src.nodata
                data = data[data != nodata] if nodata is not None else data.ravel()

                if data.size == 0:
                    msg = "No terrain data available in selected area."
                    print(msg)
                    popup = build_popup(geom, "Surface analysis", [msg])
                    add_analysis_layer(geom, color, popup)
                    next_color_index += 1
                    set_draw_color(next_draw_color())
                    return

                total_pixels = data.size
                counts = Counter(data.tolist())
                percentages = sorted(
                    ((code, (count / total_pixels) * 100) for code, count in counts.items()),
                    key=lambda item: item[1],
                    reverse=True,
                )

                lines = []
                print("Surface analysis:")
                for code, percent in percentages:
                    terrain_name = LANDCOVER_CLASSES.get(int(code), f"Class {int(code)}")
                    line = f"{terrain_name}: {percent:.2f}%"
                    lines.append(line)
                    print(line)

                popup = build_popup(geom, "Surface analysis", lines)
                add_analysis_layer(geom, color, popup)
                next_color_index += 1
                set_draw_color(next_draw_color())

        except Exception as e:
            msg = f"Error: {e}"
            print(msg)
            popup = build_popup(geom, "Surface analysis", [msg])
            add_analysis_layer(geom, color, popup)
            next_color_index += 1
            set_draw_color(next_draw_color())


draw_control.on_draw(handle_draw)

In [ ]:
# The draw handler is defined in the previous cell. Keep this cell empty so it does not override popup behavior.


In [ ]:

# app = AppLayout(
#     left_sidebar=None,
#     center=m,
#     right_sidebar=VBox([output]),
#     pane_widths=[0, 4, 3]
# )

# app
